# `Устанавливаем необходимые зависимости`

In [62]:
! pip3 install -q pyspark pyarrow parquet-tools

# `Готовим SparkContext`

Oбъект SparkContext является точкой входа для работы со Spark-кластером.

In [63]:
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext

In [64]:
sc.stop()
# Создаём конфигурационный класс с параметрами подключения
conf = (
    SparkConf()
        # Указываем URL master ноды Spark кластера
        # Можно использовать local mode, указав `local[<number_cores>]`
        # В таком случае вся обработка будет происходить на текущем компьютере
        # При этом, это может давать преимущество ввиду наличия параллелизма по ядрам компьютера
        .setMaster('local[*]')
)

# Создаём точку доступа на кластер. Позволяет использовать RDD API
sc = SparkContext(conf=conf)

# Точка доступа для использования DataFrame API
spark = SparkSession(sc)

# По завершении программы нужно обязательно выполнить остановку подключения для освобождения занятых ресурсов
# sc.stop()

# `Загрузка данных`

In [65]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/calendar.parquet

--2026-03-22 11:23:39--  https://github.com/evgpat/datasets/raw/refs/heads/main/calendar.parquet
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/calendar.parquet [following]
--2026-03-22 11:23:40--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/calendar.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24369 (24K) [application/octet-stream]
Saving to: ‘calendar.parquet.2’

calendar.parquet.2  100%[===================>]  23.80K  83.8KB/s    in 0.3s    

2026-03-22 11:23:42 (83.8 KB/s) - ‘calendar.parquet.2’ saved [24369/24369]



Атрибуты датасета **calendar.parquet**.

Датасет содержит календарные данные, которые помогают связать продажи товаров с конкретными днями, неделями и событиями.

1. **date**: Дата записи данных.

2. **wm_yr_wk**: Календарная неделя в формате года и номера недели (год-неделя).

3. **weekday**: Название дня недели.

4. **wday**: Порядковый номер дня недели.

5. **month**: Порядковый номер месяца.

6. **year**: Год наблюдения.

7. **d**: Идентификатор дня в формате последовательности (например, d_1, d_2 и т.д.).

8. **event_name_1**: Название первичного события (праздники, акции, особые события).

9. **event_type_1**: Тип первичного события (например, праздник, спортивное событие).

10. **event_name_2**: Название вторичного события (если в этот день несколько значимых событий).

11. **event_type_2**: Тип вторичного события.

12. **snap_CA, snap_TX, snap_WI**: Индикаторы (0 или 1) программы SNAP (льготная покупка продуктов) в соответствующих штатах (Калифорния, Техас, Висконсин). Показатель 1 означает, что в указанный день были доступны льготы SNAP.

In [66]:
# Считаем файл calendar.parquet и запишем данные в DataFrame с названием df_calendar
df_calendar = spark.read.parquet("calendar.parquet")
df_calendar.show(5)

+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+
|      date|wm_yr_wk|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|
+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+
|2011-01-29|   11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-01-30|   11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-01-31|   11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-02-01|   11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|
|2011-02-02|   11101|Wednesday|   5|    2|2011|d_5|        NULL|        NULL|        NULL|        NULL| 

In [67]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/sales.parquet

--2026-03-22 11:23:42--  https://github.com/evgpat/datasets/raw/refs/heads/main/sales.parquet
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/sales.parquet [following]
--2026-03-22 11:23:42--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/sales.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31179351 (30M) [application/octet-stream]
Saving to: ‘sales.parquet.2’

sales.parquet.2     100%[===================>]  29.73M  1.59MB/s    in 38s     

2026-03-22 11:24:22 (793 KB/s) - ‘sales.parquet.2’ saved [31179351/31179351]



Атрибуты датасета **sales.parquet**.

Данный датасет содержит историю продаж товаров в розничных магазинах Walmart. Используется для анализа спроса, прогнозирования продаж и оптимизации запасов.

1. **id**: Уникальный идентификатор товара в конкретном магазине.

2. **item_id**: Идентификатор товара.

3. **dept_id**: Идентификатор отдела, к которому относится товар.

4. **cat_id**: Категория товара.

5. **store_id**: Идентификатор магазина, в котором был продан товар.

6. **state_id**: Штат, в котором расположен магазин.

7. **d_1, d_2, ..., d_n**: Ежедневные данные о продажах данного товара в указанном магазине (количество проданных единиц за каждый день). Каждый атрибут соответствует одному дню, начиная с первого дня периода наблюдения.

In [68]:
# Считаем файл sales.parquet и запишем данные в DataFrame с названием df_sales
df_sales = spark.read.parquet("sales.parquet")
df_sales.show(5, vertical = True)

-RECORD 0------------------------
 id       | HOBBIES_1_001_CA_... 
 item_id  | HOBBIES_1_001        
 dept_id  | HOBBIES_1            
 cat_id   | HOBBIES              
 store_id | CA_1                 
 state_id | CA                   
 d_1      | 0                    
 d_2      | 0                    
 d_3      | 0                    
 d_4      | 0                    
 d_5      | 0                    
 d_6      | 0                    
 d_7      | 0                    
 d_8      | 0                    
 d_9      | 0                    
 d_10     | 0                    
 d_11     | 0                    
 d_12     | 0                    
 d_13     | 0                    
 d_14     | 0                    
 d_15     | 0                    
 d_16     | 0                    
 d_17     | 0                    
 d_18     | 0                    
 d_19     | 0                    
 d_20     | 0                    
 d_21     | 0                    
 d_22     | 0                    
 d_23     | 0 

In [69]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/prices.parquet

--2026-03-22 11:24:24--  https://github.com/evgpat/datasets/raw/refs/heads/main/prices.parquet
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/prices.parquet [following]
--2026-03-22 11:24:24--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/prices.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2385868 (2.3M) [application/octet-stream]
Saving to: ‘prices.parquet.2’

prices.parquet.2    100%[===================>]   2.27M   549KB/s    in 8.5s    

2026-03-22 11:24:33 (273 KB/s) - ‘prices.parquet.2’ saved [2385868/2385868]



Атрибуты датасета **prices.parquet**.

Датасет содержит информацию о динамике цен на товары в различных магазинах за определенные недели.

1. **store_id** – идентификатор магазина, в котором продаётся товар.

2. **item_id** – уникальный идентификатор конкретного товара.

3. **wm_yr_wk** – календарная неделя года, к которой относится указанная цена (в формате Walmart-календаря).

4. **sell_price** – розничная цена продажи товара в указанном магазине в течение соответствующей недели.

In [70]:
# Считаем файл prices.parquet и запишем данные в DataFrame с названием df_prices
df_prices = spark.read.parquet("prices.parquet") # ВАШ КОД
df_prices.show(5)

+--------+-------------+--------+----------+
|store_id|      item_id|wm_yr_wk|sell_price|
+--------+-------------+--------+----------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|
|    CA_1|HOBBIES_1_001|   11326|      9.58|
|    CA_1|HOBBIES_1_001|   11327|      8.26|
|    CA_1|HOBBIES_1_001|   11328|      8.26|
|    CA_1|HOBBIES_1_001|   11329|      8.26|
+--------+-------------+--------+----------+
only showing top 5 rows



# `Задачи DataFrame API`

In [71]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [72]:
# 1. Создайте новую колонку 'is_weekend' в df_calendar, которая показывает выходной день (Saturday or Saturday - 1, в ином случае 0).
# Используйте только DataFrame API и функции, импортированные из F.
# Подсказка: Используйте функции: when, isin.
df_calendar = df_calendar.withColumn("is_weekend",
                                    F.when(F.col("weekday") == "Saturday", 1)
                                    .when(F.col("weekday") == "Sunday", 1)
                                    .otherwise(0)
                                   )
df_calendar.show(5)

+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|      date|wm_yr_wk|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|is_weekend|
+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|2011-01-29|   11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-30|   11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-31|   11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         0|
|2011-02-01|   11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|         0|
|2011-02-02|   11101|Wednes

In [73]:
# 2. Переименуйте колонку 'wm_yr_wk' в 'week_id' в df_calendar
df_calendar = df_calendar.withColumnRenamed('wm_yr_wk', 'week_id')
df_calendar.show(5)

+----------+-------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|      date|week_id|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|is_weekend|
+----------+-------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|2011-01-29|  11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-30|  11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-31|  11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         0|
|2011-02-01|  11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|         0|
|2011-02-02|  11101|Wednesday|   5

In [74]:
# 3. Выведите уникальные значения weekday в df_calendar
... # ВАШ КОД
df_calendar.select("weekday").distinct().show()

+---------+
|  weekday|
+---------+
|Wednesday|
|  Tuesday|
|   Friday|
| Thursday|
| Saturday|
|   Monday|
|   Sunday|
+---------+



In [75]:
# 4. Выведите количество уникальных магазинов в df_sales
# Подсказка: используйте атрибут 'store_id'
df_prices.select("store_id").distinct().count() # ВАШ КОД

10

In [76]:
# 5. Выведите среднее количество продаж по категориям товаров за d_1
# Подсказка: используйте датасет df_sales
... # ВАШ КОД
df_sales.groupBy("cat_id").agg(
    F.avg("d_1").alias("avg_sales")
).show()

+---------+------------------+
|   cat_id|         avg_sales|
+---------+------------------+
|    FOODS|1.6129436325678497|
|HOUSEHOLD|0.5433619866284622|
|  HOBBIES|0.6661946902654867|
+---------+------------------+



In [77]:
# 6. Создайте новый атрибут 'sell_price_int' в df_prices, изменив тип данных атрибута 'sell_price' на Integer
# Подсказка, для изменения типа используйте метод cast
df_prices = df_prices.withColumn('sell_price_int', F.col('sell_price').cast("int")) # ВАШ КОД
df_prices.show(5)

+--------+-------------+--------+----------+--------------+
|store_id|      item_id|wm_yr_wk|sell_price|sell_price_int|
+--------+-------------+--------+----------+--------------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|             9|
|    CA_1|HOBBIES_1_001|   11326|      9.58|             9|
|    CA_1|HOBBIES_1_001|   11327|      8.26|             8|
|    CA_1|HOBBIES_1_001|   11328|      8.26|             8|
|    CA_1|HOBBIES_1_001|   11329|      8.26|             8|
+--------+-------------+--------+----------+--------------+
only showing top 5 rows



In [78]:
# 7. Выведите 5 самых дорогих товаров в df_prices
... # ВАШ КОД
df_prices.orderBy(F.col('sell_price_int').desc()).show(5)

+--------+---------------+--------+----------+--------------+
|store_id|        item_id|wm_yr_wk|sell_price|sell_price_int|
+--------+---------------+--------+----------+--------------+
|    WI_3|HOUSEHOLD_2_406|   11317|    107.32|           107|
|    WI_3|HOUSEHOLD_2_406|   11318|    107.32|           107|
|    WI_3|HOUSEHOLD_2_406|   11319|    107.32|           107|
|    WI_2|HOUSEHOLD_2_406|   11241|     61.46|            61|
|    WI_2|HOUSEHOLD_2_406|   11246|     61.46|            61|
+--------+---------------+--------+----------+--------------+
only showing top 5 rows



In [79]:
# 8. Используя sql, найдите среднее количество продаж по штатам (назвав атрибут 'avg_sales') для d1
df_sales.createOrReplaceTempView("sales")
spark.sql("select state_id, avg(d_1) as avg_sales from sales group by state_id").show(5) # ВАШ КОД

+--------+------------------+
|state_id|         avg_sales|
+--------+------------------+
|      CA|1.1639061987536898|
|      TX|1.0318137094129223|
|      WI|0.9837105061768886|
+--------+------------------+



In [80]:
# 9. Выведите 10 строк датасета с товарами, проданных в Техасе
# Подсказка: используйте датасет df_sales
... # ВАШ КОД
df_sales.filter(F.col("state_id") == "TX").show(10)

+--------------------+-------------+---------+-------+--------+--------+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----

In [81]:
# 10. Выведите 5 самых высоких цен на товары (sell_price), которые были в период с "2016-01-01" по "2016-01-31"
# Подсказка: используйте join, используйте метод between, используйте метод distinct() для уникальности цен
df_joined = df_sales.join(df_prices, on = "item_id")
df_joined = df_joined.join(df_calendar, df_calendar["week_id"] == df_joined["wm_yr_wk"], how="inner")
df_joined_filtered = df_joined.filter((F.col("year") == 2016) & (F.col("month")==1))
distinct_sell_price = df_joined_filtered.select("sell_price").distinct()
distinct_sell_price.orderBy(F.col('sell_price').desc()).show(5)

+----------+
|sell_price|
+----------+
|     29.97|
|     29.96|
|     28.96|
|     27.98|
|     26.98|
+----------+
only showing top 5 rows



In [82]:
# 11. Найдите товары с наибольшими продажами в каждой категории (cat_id) за d_1, используя датасет df_sales
# Подсказка: используйте оконную функцию rank()
window_spec = Window.partitionBy("cat_id").orderBy(F.col("d_1").desc()) # ВАШ КОД
df_sales = df_sales.withColumn("rank", F.rank().over(window_spec))
df_sales.filter(F.col("rank") == 1).show()

+--------------------+---------------+-----------+---------+--------+--------+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----

In [83]:
# 12. Создайте новую колонку "price_category" в df_prices, указав высокая или низкая цена: если цена больше 5, то "High", иначе "Low"
# Подсказка: используйте конструкцию when().otherwise
... # ВАШ КОД
df_prices = df_prices.withColumn("price_category",
                                    F.when(F.col("sell_price_int") > 5, 'High')
                                    .otherwise('Low')
                                   )
df_prices.show(10)

+--------+-------------+--------+----------+--------------+--------------+
|store_id|      item_id|wm_yr_wk|sell_price|sell_price_int|price_category|
+--------+-------------+--------+----------+--------------+--------------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|             9|          High|
|    CA_1|HOBBIES_1_001|   11326|      9.58|             9|          High|
|    CA_1|HOBBIES_1_001|   11327|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11328|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11329|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11330|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11331|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11332|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11333|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11334|      8.26|             8|          High|
+--------+-------------+-

In [84]:
# 13. Вычислите суммарные продажи по всем магазинам (store_id) за d_1
... # ВАШ КОД
df_sales.createOrReplaceTempView("sales")
spark.sql("select store_id, sum(d_1) as avg_sales from sales group by store_id").show(10) # ВАШ КОД

+--------+---------+
|store_id|avg_sales|
+--------+---------+
|    TX_2|     3852|
|    TX_1|     2556|
|    CA_4|     1625|
|    CA_2|     3494|
|    CA_1|     4337|
|    CA_3|     4739|
|    WI_2|     2256|
|    WI_3|     4038|
|    WI_1|     2704|
|    TX_3|     3030|
+--------+---------+



# `Задачи RDD`

### Данные

Файл - `transaction.csv`

Формат записей:

```
user_id, timestamp, item_id, category, price, quantity, city
```





In [98]:
# Задание 0. Загрузите данные в RDD
rdd = spark.sparkContext.textFile("transactions.csv") # ВАШ КОД
... # ВАШ КОД
rdd.take(5)

['user_id,timestamp,item_id,category,price,quantity,city',
 '22,2025-03-07 15:41,E22,beauty,1227,1,Perm',
 '45,2025-03-19 20:27,Q77,auto,98,1,Kazan',
 '37,2025-03-04 14:55,B07,books,1435,1,Rostov',
 '52,2025-03-16 23:39,Q77,auto,630,1,Kazan']

In [99]:
header = rdd.first()
data_rdd = rdd.filter(lambda line: line != header)
def parse_and_compute(line):
    parts = line.split(",")
    return (parts[0], parts[1], parts[2], parts[3], 
            float(parts[4]), int(parts[5]), parts[6], 
            float(parts[4]) * int(parts[5]))

rdd = data_rdd.map(parse_and_compute)
for record in rdd.take(5):
    print(record)

('22', '2025-03-07 15:41', 'E22', 'beauty', 1227.0, 1, 'Perm', 1227.0)
('45', '2025-03-19 20:27', 'Q77', 'auto', 98.0, 1, 'Kazan', 98.0)
('37', '2025-03-04 14:55', 'B07', 'books', 1435.0, 1, 'Rostov', 1435.0)
('52', '2025-03-16 23:39', 'Q77', 'auto', 630.0, 1, 'Kazan', 630.0)
('69', '2025-03-06 10:52', 'E22', 'beauty', 1248.0, 4, 'Novosibirsk', 4992.0)


In [102]:
# Задание 1. Найти ТОП-5 категорий по общей выручке

category_revenue = rdd.map(lambda x: (x[3], x[7])) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(5) # ВАШ КОД

for x in category_revenue:
    print(x)

('beauty', 453070.0)
('home', 444245.0)
('kids', 403922.0)
('sport', 389754.0)
('toys', 381717.0)


In [104]:
unique__by_category 

In [111]:
# Задание 2. Найти пользователей, покупавших в более чем двух категориях

users_many = rdd.map(lambda x: (x[0], x[3])) \
    .distinct() \
    .map(lambda x: (x[0], 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[1] > 2) \
    .collect()
for u in users_many[:10]:
     print(u[0], " → ", u[1], "категорий:", u[1])

22  →  6 категорий: 6
45  →  6 категорий: 6
62  →  7 категорий: 7
79  →  8 категорий: 8
18  →  8 категорий: 8
64  →  9 категорий: 9
44  →  5 категорий: 5
13  →  8 категорий: 8
21  →  9 категорий: 9
15  →  7 категорий: 7


In [112]:
# Задание 3. Найти средний чек по каждому городу

avg_check_by_city = rdd.map(lambda x: (x[6], (x[7], 1))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda x: round(x[0] / x[1], 2)) \
    .sortBy(lambda x: x[1], ascending=False) \
    .collect()

for x in avg_check_by_city:
    print(x)

('Ekaterinburg', 2639.17)
('Ufa', 2435.83)
('Kazan', 2366.48)
('Rostov', 2353.31)
('Moscow', 2326.26)
('Novosibirsk', 2292.32)
('Samara', 2219.05)
('Perm', 2216.31)
('SPB', 2148.12)


In [117]:
# Задание 4. Найти самый продаваемый товар внутри каждой категории и
# суммарное количество проданных единиц этого товара

top_product_by_category = rdd.map(lambda x: ((x[3], x[2]), x[5])) \
    .reduceByKey(lambda a, b: a + b) \
    .map(lambda x: (x[0][0], (x[0][1], x[1]))) \
    .groupByKey() \
    .mapValues(lambda items: max(items, key=lambda x: x[1])) \
    .collect()

for x in top_product_by_category:
    print(x)

('toys', ('D14', 493))
('beauty', ('E22', 568))
('auto', ('Q77', 452))
('kids', ('H55', 517))
('garden', ('G10', 448))
('sport', ('C02', 476))
('home', ('F99', 590))
('books', ('B07', 465))
('electronics', ('A13', 475))


In [119]:
from datetime import datetime
def extract_hour(ts):
    dt = datetime.strptime(ts, "%Y-%m-%d %H:%M")
    return dt.hour

hour_rev = rdd.map(lambda x: (extract_hour(x[1]), x[7])) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()

peak_hour, peak_revenue = max(hour_rev, key=lambda x: x[1])

total_revenue = rdd.map(lambda x: x[7]).sum()

peak_share = peak_revenue / total_revenue

print("Час пик:", peak_hour)
print("Выручка в час пик:", peak_revenue)
print("Общая выручка:", total_revenue)
print("Доля общего оборота:", f"{peak_share:.2%}")

Час пик: 11
Выручка в час пик: 181001.0
Общая выручка: 3497715.0
Доля общего оборота: 5.17%
